In [19]:
ed = pd.read_csv("train.csv")

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
df = pd.read_csv('train.csv')
#####################


# --- 1. Préparation ---
# On garde les colonnes telles quelles (CatBoost gère les strings)
categorical_features = ['type_contrat', 'utilisation', 'essence_vehicule', 'code_postal', 'marque_vehicule']

numerical_features = ['bonus', 'duree_contrat', 'age_conducteur1', 'anciennete_permis1', 'prix_vehicule', 
                      'vitesse_vehicule', 'cylindre_vehicule', 'anciennete_vehicule', 'din_vehicule', 'poids_vehicule']

# Remplissage simple pour les numériques (CatBoost gère aussi les NaNs, mais c'est plus propre)
df['anciennete_vehicule'] = df['anciennete_vehicule'].fillna(-1)

# --- 2. MODÈLE FRÉQUENCE (Probabilité de sinistre) ---
X = df[numerical_features + categorical_features]
y_freq = (df['nombre_sinistres'] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y_freq, test_size=0.2, random_state=8)

model_freq = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    cat_features=categorical_features, # On lui donne directement la liste
    verbose=100
)

model_freq.fit(X_train, y_train, eval_set=(X_test, y_test))

# --- 3. MODÈLE SÉVÉRITÉ (Montant moyen) ---
# Uniquement sur les lignes avec un montant > 0
df_claims = df[df['montant_sinistre'] > 0].copy()
X_sev = df_claims[numerical_features + categorical_features]
y_sev = df_claims['montant_sinistre']

model_sev = CatBoostRegressor(
    iterations=500,
    objective='Tweedie:variance_power=1.5', # Très efficace pour les montants d'assurance
    cat_features=categorical_features,
    verbose=100
)

model_sev.fit(X_sev, y_sev)

# --- 4. CALCUL DU COÛT ESPÉRÉ ---
df['probabilite'] = model_freq.predict_proba(X)[:, 1]
df['montant_potentiel'] = model_sev.predict(X)
df['pred'] = df['probabilite'] * df['montant_potentiel']

0:	learn: 0.6567674	test: 0.6567159	best: 0.6567159 (0)	total: 103ms	remaining: 51.6s
100:	learn: 0.2152067	test: 0.2145559	best: 0.2145559 (100)	total: 7.53s	remaining: 29.7s
200:	learn: 0.2111496	test: 0.2127170	best: 0.2127170 (200)	total: 17.2s	remaining: 25.7s
300:	learn: 0.2092157	test: 0.2125586	best: 0.2125257 (280)	total: 26.7s	remaining: 17.6s
400:	learn: 0.2071674	test: 0.2125129	best: 0.2124523 (347)	total: 36s	remaining: 8.88s
499:	learn: 0.2050426	test: 0.2124850	best: 0.2123956 (457)	total: 45.1s	remaining: 0us

bestTest = 0.2123955907
bestIteration = 457

Shrink model to first 458 iterations.
0:	learn: 3441.1353526	total: 28.2ms	remaining: 14.1s
100:	learn: 239.4181838	total: 2.76s	remaining: 10.9s
200:	learn: 168.0099872	total: 5.39s	remaining: 8.02s
300:	learn: 167.1882955	total: 8.81s	remaining: 5.82s
400:	learn: 166.5614727	total: 12.4s	remaining: 3.05s
499:	learn: 165.9833895	total: 15.9s	remaining: 0us


In [ ]:
import pandas as pd

# 1. Charger le nouveau fichier
nouveau_df = pd.read_csv('test.csv')

# 2. Prétraitement identique (très important !)
# On remplit les valeurs manquantes comme pour l'entraînement
nouveau_df['anciennete_vehicule'] = nouveau_df['anciennete_vehicule'].fillna(nouveau_df['anciennete_vehicule'].median())

# 3. Sélectionner les colonnes utilisées par le modèle
# (Assurez-vous que numerical_features et categorical_features sont bien définis)
X_nouveau = nouveau_df[numerical_features + categorical_features]

# 4. Appliquer les prédictions
# Probabilité de sinistre (Modèle 1)
nouveau_df['probabilite'] = model_freq.predict_proba(X_nouveau)[:, 1]

# Montant estimé si sinistre il y a (Modèle 2)
nouveau_df['montant_potentiel'] = model_sev.predict(X_nouveau)

# Coût espéré (Prime pure)
nouveau_df['pred'] = nouveau_df['probabilite'] * nouveau_df['montant_potentiel']*1.2

# 5. Sauvegarder les résultats
nouveau_df.to_csv('resultats_predictions.csv', index=False)
print("Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'")

Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'


In [34]:
pp=pd.read_csv('resultats_predictions.csv')
df_final = pp[['index', 'pred']]

# 3. Sauvegarder le résultat dans un nouveau fichier CSV
# index=False permet de ne pas rajouter une colonne de numérotation inutile
df_final.to_csv('resultat_final2.csv', index=False)

print("Le fichier filtré a été sauvegardé sous le nom 'resultat_final.csv'")

Le fichier filtré a été sauvegardé sous le nom 'resultat_final.csv'
